# COX PH (Cox Proportional Hazard)
- PH assumption:
    - The effect (hazard ratio) of each feature stays constant over time.
- Detecting nonlinearity in relationship between the log hazard and the covariates.
- If a variable’s influence changes with time (e.g., being older increases risk early but not later), the PH assumption is violated (SO IT REQUIRES SPECIAL SOLUTIONS)

In short, studies using the **FLamby Fed-TCGA-BRCA** dataset follow a clear analysis pipeline. They use a clean set of clinical features (mostly one-hot encoded) and split the data by hospital to simulate a federated setup. Researchers first looked at **Kaplan–Meier curves** and **log-rank tests** to see how survival differed across sites. The **Cox proportional hazards model** was the main baseline, evaluated with the **concordance index**. **They noted that the proportional hazards assumption might not always hold between centers since some survival curves were clearly different, but they didn’t apply formal corrections like stratification.**


Instead, later work tried more flexible models such as **DeepSurv**, **DeepHit**, and **FedSurF++**, which don’t rely on that assumption. Overall, these studies outline how to preprocess, analyze, and train survival models in a federated setup using this dataset.


Maybe then should I try to apply a formal correction?

In [1]:
from lifelines import CoxPHFitter
import pandas as pd
import torch
from flamby.datasets.fed_tcga_brca import FedTcgaBrca

In [2]:
# Prints info of all centers in train and test (since they are already splited)
all_centers = []

for center in range(6):
    # Load both train and test
    for mode in [True, False]:  # True=train, False=test
        dataset = FedTcgaBrca(center=center, train=mode)
        X_list, y_list = [], []

        for X, y in dataset:
            X_list.append(X.numpy())
            y_list.append(y.numpy())

        df_X = pd.DataFrame(X_list)
        df_y = pd.DataFrame(y_list, columns=["event", "time"])
        df_center = pd.concat([df_X, df_y], axis=1)
        df_center["center"] = center
        df_center["split"] = "train" if mode else "test"  # optional column
        all_centers.append(df_center)

# combine into single dataframe
df_all = pd.concat(all_centers, ignore_index=True)
df_all.columns = [f"feature_{i}" for i in range(df_all.shape[1]-4)] + ["event", "time", "center", "split"]


/Users/nataliamorenoblasco/Desktop/Master_Thesis/FLamby/flamby/datasets/fed_tcga_brca/dataset.py:51: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  return (torch.tensor(x, dtype=self.X_dtype), torch.tensor(y, dtype=self.y_dtype))
/Users/nataliamorenoblasco/Desktop/Master_Thesis/FLamby/flamby/datasets/fed_tcga_brca/dataset.py:51: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  return (torch.tensor(x, dtype=self.X_dtype), torch.tensor(y, dtype=self.y_dtype))
/Users/nataliamorenoblasco/Desktop/Master_Thesis/FLamby/flamby/datasets/fed_tcga_brca/dataset.py:51: FutureWarning: Series.__getitem__ treating keys as positions is d

In [3]:
from lifelines import CoxPHFitter
from lifelines.statistics import proportional_hazard_test

# Exclude columns:
columns_to_exclude = ['center', 'split']
numeric_df = df_all.drop(columns=columns_to_exclude)

# Fit the Cox Proportional Hazards model
cph = CoxPHFitter()
cph.fit(numeric_df, duration_col='time', event_col='event')

# Print the summary of the model
cph.print_summary()

# Check the proportional hazards assumption
cph.check_assumptions(numeric_df, p_value_threshold=0.05)

# Alternatively, use proportional_hazard_test for a specific test
results = proportional_hazard_test(cph, numeric_df, time_transform="rank")
print(results.summary)

ConvergenceError: Convergence halted due to matrix inversion problems. Suspicion is high collinearity. Please see the following tips in the lifelines documentation: https://lifelines.readthedocs.io/en/latest/Examples.html#problems-with-convergence-in-the-cox-proportional-hazard-modelMatrix is singular.

## Variance Inflation Factor (VIF):

- VIF quantifies how much the variance of a regression coefficient is inflated due to multicollinearity.
- Features with VIF > 10 are typically considered highly collinear.

In [5]:
from statsmodels.stats.outliers_influence import variance_inflation_factor
import numpy as np

# Calculate VIF for each feature
X = numeric_df.drop(columns=['time', 'event'])  # Exclude time and event columns
vif_data = pd.DataFrame()
vif_data["feature"] = X.columns
vif_data["VIF"] = [variance_inflation_factor(X.values, i) for i in range(X.shape[1])]

# Display features with high VIF
print(vif_data)

# Drop features with high VIF (e.g., VIF > 10)
high_vif_features = vif_data[vif_data["VIF"] > 10]["feature"].tolist()
reduced_df = numeric_df.drop(columns=high_vif_features)

# Refit the Cox Proportional Hazards model
cph = CoxPHFitter()
cph.fit(reduced_df, duration_col='time', event_col='event')

# Print the summary of the model
cph.print_summary()

# Check the proportional hazards assumption
cph.check_assumptions(reduced_df, p_value_threshold=0.05)

       feature           VIF
0    feature_0  2.345658e+01
1    feature_1  2.542006e+01
2    feature_2  6.882979e+00
3    feature_3  1.037843e+01
4    feature_4  2.918926e+01
5    feature_5  1.774372e+01
6    feature_6  1.168634e+02
7    feature_7  1.458689e+00
8    feature_8  1.539354e+00
9    feature_9  1.956785e+00
10  feature_10  2.233033e+00
11  feature_11  2.561203e+00
12  feature_12  2.905530e+00
13  feature_13  8.042142e+13
14  feature_14  4.170000e+13
15  feature_15  2.370316e+14
16  feature_16  1.407375e+14
17  feature_17  1.916425e+14
18  feature_18  3.805514e+00
19  feature_19  1.254599e+01
20  feature_20  4.448885e+00
21  feature_21  1.640277e+00
22  feature_22  4.531973e+00
23  feature_23  4.369691e+00
24  feature_24  9.672302e+01
25  feature_25  3.631935e+13
26  feature_26  3.602880e+14
27  feature_27  3.631935e+13
28  feature_28  3.602880e+14
29  feature_29  6.900808e+01
30  feature_30  6.917087e+01
31  feature_31  2.140933e+00
32  feature_32  7.561660e+00
33  feature_33

<lifelines.CoxPHFitter: fitted with 1088 total observations, 937 right-censored observations>
             duration col = 'time'
                event col = 'event'
      baseline estimation = breslow
   number of observations = 1088
number of events observed = 151
   partial log-likelihood = -787.62
         time fit was run = 2025-10-24 08:36:29 UTC

---
            coef exp(coef)  se(coef)  coef lower 95%  coef upper 95% exp(coef) lower 95% exp(coef) upper 95%
covariate                                                                                                   
feature_2  -0.38      0.68      0.35           -1.07            0.30                0.34                1.35
feature_7  -0.43      0.65      0.31           -1.04            0.19                0.35                1.21
feature_8  -1.05      0.35      0.32           -1.68           -0.42                0.19                0.65
feature_9  -0.49      0.61      0.28           -1.04            0.06                0.35                1.06
feature_10 -0.66      0.51      0.32           -1.28           -0.04                0.28                0.96
feature_11  0.18      1.20      0.34           -0.49            0.85                0.61                2.34
feature_12  0.41      1.51      0.34           -0.25            1.07                0.78                2.91
feature_18 -0.24      0.79      0.25           -0.73            0.25                0.48                1.28
feature_20  0.22      1.25      0.23           -0.23            0.68                0.79                1.97
feature_21 -0.56      0.57      0.31           -1.16            0.04                0.32                1.04
feature_22 -0.58      0.56      0.24           -1.05           -0.12                0.35                0.89
feature_23  0.28      1.32      0.29           -0.29            0.85                0.74                2.34
feature_31  0.93      2.53      0.34            0.27            1.58                1.31                4.88
feature_32 -1.15      0.32      0.26           -1.66           -0.63                0.19                0.53
feature_33 -0.67      0.51      0.33           -1.32           -0.02                0.27                0.98

            cmp to     z      p  -log2(p)
covariate                                
feature_2     0.00 -1.10   0.27      1.88
feature_7     0.00 -1.36   0.17      2.53
feature_8     0.00 -3.29 <0.005      9.94
feature_9     0.00 -1.75   0.08      3.64
feature_10    0.00 -2.10   0.04      4.81
feature_11    0.00  0.53   0.60      0.74
feature_12    0.00  1.22   0.22      2.17
feature_18    0.00 -0.97   0.33      1.59
feature_20    0.00  0.96   0.33      1.58
feature_21    0.00 -1.82   0.07      3.88
feature_22    0.00 -2.46   0.01      6.17
feature_23    0.00  0.95   0.34      1.55
feature_31    0.00  2.77   0.01      7.47
feature_32    0.00 -4.38 <0.005     16.38
feature_33    0.00 -2.03   0.04      4.57
---
Concordance = 0.75
Partial AIC = 1605.23
log-likelihood ratio test = 134.94 on 15 df
-log2(p) of ll-ratio test = 68.57

The ``p_value_threshold`` is set at 0.05. Even under the null hypothesis of no violations, some
covariates will be below the threshold by chance. This is compounded when there are many covariates.
Similarly, when there are lots of observations, even minor deviances from the proportional hazard
assumption will be flagged.

With that in mind, it's best to use a combination of statistical tests and visual tests to determine
the most serious violations. Produce visual plots using ``check_assumptions(..., show_plots=True)``
and looking for non-constant lines. See link [A] below for a full example.



<lifelines.StatisticalResult: proportional_hazard_test>
 null_distribution = chi squared
degrees_of_freedom = 1
             model = <lifelines.CoxPHFitter: fitted with 1088 total observations, 937 right-censored observations>
         test_name = proportional_hazard_test

---
                 test_statistic    p  -log2(p)
feature_10 km              0.01 0.94      0.10
           rank            0.21 0.65      0.62
feature_11 km              0.04 0.84      0.26
           rank            0.01 0.93      0.10
feature_12 km              1.59 0.21      2.27
           rank            2.31 0.13      2.96
feature_18 km              1.26 0.26      1.93
           rank            0.91 0.34      1.56
feature_2  km              0.25 0.62      0.69
           rank            0.25 0.62      0.69
feature_20 km              0.26 0.61      0.71
           rank            0.11 0.74      0.44
feature_21 km              0.23 0.64      0.65
           rank            0.14 0.71      0.49
feature_22 km              0.01 0.91      0.14
           rank            0.06 0.80      0.32
feature_23 km              0.45 0.50      0.99
           rank            0.30 0.58      0.78
feature_31 km              3.65 0.06      4.16
           rank            4.56 0.03      4.93
feature_32 km              0.20 0.65      0.62
           rank            0.38 0.54      0.89
feature_33 km              1.04 0.31      1.70
           rank            0.53 0.47      1.10
feature_7  km              1.31 0.25      1.98
           rank            1.87 0.17      2.54
feature_8  km              2.36 0.12      3.01
           rank            3.12 0.08      3.69
feature_9  km              0.12 0.73      0.45
           rank            0.00 0.98      0.03



1. Variable 'feature_31' failed the non-proportional test: p-value is 0.0327.

   Advice: with so few unique values (only 2), you can include `strata=['feature_31', ...]` in the
call in `.fit`. See documentation in link [E] below.

---
[A]  https://lifelines.readthedocs.io/en/latest/jupyter_notebooks/Proportional%20hazard%20assumption.html
[B]  https://lifelines.readthedocs.io/en/latest/jupyter_notebooks/Proportional%20hazard%20assumption.html#Bin-variable-and-stratify-on-it
[C]  https://lifelines.readthedocs.io/en/latest/jupyter_notebooks/Proportional%20hazard%20assumption.html#Introduce-time-varying-covariates
[D]  https://lifelines.readthedocs.io/en/latest/jupyter_notebooks/Proportional%20hazard%20assumption.html#Modify-the-functional-form
[E]  https://lifelines.readthedocs.io/en/latest/jupyter_notebooks/Proportional%20hazard%20assumption.html#Stratification



[]

- INTERPRETATION OF VIF(Variance Inflation Factor):
    - 

In [6]:
# Save the VIF data to a CSV file
vif_data.to_csv('vif_results.csv', index=False)
print("VIF results saved to 'vif_results.csv'")

VIF results saved to 'vif_results.csv'


In [7]:
# Remove features with high VIF values
features_to_remove = ['feature_13', 'feature_14', 'feature_15', 'feature_16', 'feature_17',
                      'feature_25', 'feature_26', 'feature_27', 'feature_28', 'feature_34',
                      'feature_35', 'feature_36', 'feature_37', 'feature_38']

# Drop the identified features from the dataset
final_df = numeric_df.drop(columns=features_to_remove)

# Refit the Cox Proportional Hazards model
cph = CoxPHFitter()
cph.fit(final_df, duration_col='time', event_col='event')

# Print the summary of the model
cph.print_summary()

# Check the proportional hazards assumption
cph.check_assumptions(final_df, p_value_threshold=0.05)

<lifelines.CoxPHFitter: fitted with 1088 total observations, 937 right-censored observations>
             duration col = 'time'
                event col = 'event'
      baseline estimation = breslow
   number of observations = 1088
number of events observed = 151
   partial log-likelihood = -777.03
         time fit was run = 2025-10-24 08:57:25 UTC

---
             coef exp(coef)  se(coef)  coef lower 95%  coef upper 95% exp(coef) lower 95% exp(coef) upper 95%
covariate                                                                                                    
feature_0    0.02      1.02      0.01            0.01            0.04                1.01                1.04
feature_1    1.19      3.29      1.02           -0.81            3.19                0.44               24.38
feature_2    0.36      1.43      1.12           -1.83            2.55                0.16               12.79
feature_3   12.10  1.79e+05   2360.44        -4614.29         4638.48                0.00                 inf
feature_4   12.22  2.03e+05   2360.44        -4614.16         4638.61                0.00                 inf
feature_5   12.56  2.85e+05   2360.44        -4613.83         4638.94                0.00                 inf
feature_6   11.98  1.60e+05   2360.44        -4614.40         4638.37                0.00                 inf
feature_7   -0.42      0.66      0.32           -1.05            0.21                0.35                1.23
feature_8   -1.09      0.34      0.33           -1.73           -0.44                0.18                0.64
feature_9   -0.35      0.71      0.29           -0.91            0.21                0.40                1.24
feature_10  -0.65      0.52      0.32           -1.29           -0.02                0.28                0.98
feature_11   0.33      1.39      0.35           -0.35            1.01                0.71                2.75
feature_12   0.38      1.47      0.34           -0.28            1.05                0.76                2.85
feature_18  -0.57      0.57      0.30           -1.16            0.03                0.31                1.03
feature_19  -0.27      0.76      0.26           -0.78            0.23                0.46                1.26
feature_20  -0.06      0.94      0.31           -0.67            0.55                0.51                1.73
feature_21  -0.48      0.62      0.32           -1.10            0.15                0.33                1.16
feature_22  -0.57      0.56      0.24           -1.05           -0.10                0.35                0.91
feature_23   0.16      1.18      0.30           -0.42            0.75                0.66                2.11
feature_24  -0.55      0.58      0.45           -1.43            0.34                0.24                1.40
feature_29  12.23  2.04e+05   2259.62        -4416.55         4441.00                0.00                 inf
feature_30 -11.66      0.00   2259.62        -4440.43         4417.12                0.00                 inf
feature_31   0.92      2.51      0.35            0.23            1.61                1.25                5.02
feature_32  -0.85      0.43      0.27           -1.39           -0.32                0.25                0.73
feature_33  -0.92      0.40      0.36           -1.63           -0.22                0.20                0.80

            cmp to     z      p  -log2(p)
covariate                                
feature_0     0.00  3.49 <0.005     11.03
feature_1     0.00  1.17   0.24      2.04
feature_2     0.00  0.32   0.75      0.42
feature_3     0.00  0.01   1.00      0.01
feature_4     0.00  0.01   1.00      0.01
feature_5     0.00  0.01   1.00      0.01
feature_6     0.00  0.01   1.00      0.01
feature_7     0.00 -1.32   0.19      2.42
feature_8     0.00 -3.31 <0.005     10.07
feature_9     0.00 -1.22   0.22      2.17
feature_10    0.00 -2.02   0.04      4.54
feature_11    0.00  0.96   0.34      1.57
feature_12    0.00  1.14   0.26      1.97
feature_18    0.00 -1.87   0.06      4.0

The ``p_value_threshold`` is set at 0.05. Even under the null hypothesis of no violations, some
covariates will be below the threshold by chance. This is compounded when there are many covariates.
Similarly, when there are lots of observations, even minor deviances from the proportional hazard
assumption will be flagged.

With that in mind, it's best to use a combination of statistical tests and visual tests to determine
the most serious violations. Produce visual plots using ``check_assumptions(..., show_plots=True)``
and looking for non-constant lines. See link [A] below for a full example.



<lifelines.StatisticalResult: proportional_hazard_test>
 null_distribution = chi squared
degrees_of_freedom = 1
             model = <lifelines.CoxPHFitter: fitted with 1088 total observations, 937 right-censored observations>
         test_name = proportional_hazard_test

---
                 test_statistic      p  -log2(p)
feature_0  km              0.64   0.42      1.23
           rank            3.02   0.08      3.61
feature_1  km              1.72   0.19      2.40
           rank            2.58   0.11      3.21
feature_10 km              0.05   0.83      0.27
           rank            0.01   0.92      0.11
feature_11 km              0.10   0.75      0.41
           rank            0.33   0.56      0.83
feature_12 km              0.86   0.35      1.50
           rank            1.48   0.22      2.16
feature_18 km              9.18 <0.005      8.67
           rank            7.47   0.01      7.31
feature_19 km             15.44 <0.005     13.52
           rank           13.75 <0.005     12.23
feature_2  km              1.49   0.22      2.17
           rank            2.15   0.14      2.81
feature_20 km              7.07   0.01      6.99
           rank            3.35   0.07      3.90
feature_21 km              0.12   0.73      0.46
           rank            0.01   0.93      0.10
feature_22 km              0.47   0.50      1.01
           rank            0.55   0.46      1.13
feature_23 km              0.08   0.77      0.37
           rank            0.12   0.73      0.46
feature_24 km              0.95   0.33      1.60
           rank            1.71   0.19      2.39
feature_29 km              0.00   1.00      0.00
           rank            0.00   1.00      0.00
feature_3  km              0.00   1.00      0.00
           rank            0.00   1.00      0.00
feature_30 km              0.00   1.00      0.00
           rank            0.00   1.00      0.00
feature_31 km              1.73   0.19      2.41
           rank            1.70   0.19      2.37
feature_32 km              0.05   0.82      0.28
           rank            0.06   0.81      0.30
feature_33 km              5.14   0.02      5.42
           rank            4.34   0.04      4.75
feature_4  km              0.00   1.00      0.00
           rank            0.00   1.00      0.00
feature_5  km              0.00   1.00      0.00
           rank            0.00   1.00      0.00
feature_6  km              0.00   1.00      0.00
           rank            0.00   1.00      0.00
feature_7  km              5.64   0.02      5.83
           rank            7.54   0.01      7.37
feature_8  km              3.66   0.06      4.17
           rank            5.07   0.02      5.36
feature_9  km              0.27   0.60      0.73
           rank            0.01   0.93      0.11



1. Variable 'feature_7' failed the non-proportional test: p-value is 0.0060.

   Advice: with so few unique values (only 2), you can include `strata=['feature_7', ...]` in the
call in `.fit`. See documentation in link [E] below.

2. Variable 'feature_8' failed the non-proportional test: p-value is 0.0244.

   Advice: with so few unique values (only 2), you can include `strata=['feature_8', ...]` in the
call in `.fit`. See documentation in link [E] below.

3. Variable 'feature_18' failed the non-proportional test: p-value is 0.0025.

   Advice: with so few unique values (only 2), you can include `strata=['feature_18', ...]` in the
call in `.fit`. See documentation in link [E] below.

4. Variable 'feature_19' failed the non-proportional test: p-value is 0.0001.

   Advice: with so few unique values (only 2), you can include `strata=['feature_19', ...]` in the
call in `.fit`. See documentation in link [E] below.

5. Variable 'feature_20' failed the non-proportional test: p-value is 0.00

[]

In [11]:
# Stratify on 'feature_31', 'feature_7', 'feature_8', 'feature_18', 'feature_19', and 'feature_33', and bin 'feature_0' to handle non-proportionality
# Bin 'feature_0' into categories
final_df['feature_0_binned'] = pd.cut(final_df['feature_0'], bins=5, labels=False)

# Fit the Cox Proportional Hazards model with stratification
cph = CoxPHFitter()
cph.fit(final_df, duration_col='time', event_col='event', strata=['feature_31', 'feature_7', 'feature_8', 'feature_18', 'feature_19', 'feature_33', 'feature_0_binned'])

# Print the summary of the model
cph.print_summary()

# Check the proportional hazards assumption again
cph.check_assumptions(final_df, p_value_threshold=0.05)

<lifelines.CoxPHFitter: fitted with 1088 total observations, 937 right-censored observations>
             duration col = 'time'
                event col = 'event'
                   strata = ['feature_31', 'feature_7', 'feature_8', 'feature_18', 'feature_19', 'feature_33', 'feature_0_binned']
      baseline estimation = breslow
   number of observations = 1088
number of events observed = 151
   partial log-likelihood = -266.73
         time fit was run = 2025-10-24 09:03:03 UTC

---
             coef exp(coef)  se(coef)  coef lower 95%  coef upper 95% exp(coef) lower 95% exp(coef) upper 95%
covariate                                                                                                    
feature_0    0.02      1.02      0.03           -0.04            0.07                0.96                1.07
feature_1    0.81      2.24      1.04           -1.23            2.84                0.29               17.19
feature_2   -0.17      0.84      1.16           -2.45            2.10                0.09                8.17
feature_3   14.46  1.90e+06   9145.82       -17911.02        17939.94                0.00                 inf
feature_4   15.70  6.55e+06   9145.82       -17909.78        17941.17                0.00                 inf
feature_5   15.63  6.16e+06   9145.82       -17909.84        17941.11                0.00                 inf
feature_6   15.02  3.33e+06   9145.82       -17910.46        17940.49                0.00                 inf
feature_9   -0.52      0.59      0.34           -1.19            0.14                0.31                1.15
feature_10  -0.78      0.46      0.36           -1.49           -0.07                0.22                0.93
feature_11   0.30      1.35      0.38           -0.44            1.04                0.64                2.82
feature_12   0.36      1.43      0.38           -0.40            1.11                0.67                3.03
feature_20  -0.56      0.57      0.38           -1.31            0.18                0.27                1.20
feature_21  -1.17      0.31      0.49           -2.13           -0.21                0.12                0.81
feature_22  -0.63      0.53      0.27           -1.16           -0.09                0.31                0.92
feature_23  -0.16      0.85      0.34           -0.82            0.50                0.44                1.65
feature_24  -0.23      0.79      0.67           -1.54            1.07                0.21                2.93
feature_29  14.34  1.69e+06   3366.94        -6584.74         6613.42                0.00                 inf
feature_30 -14.08      0.00   3366.94        -6613.16         6585.00                0.00                 inf
feature_32  -0.91      0.40      0.33           -1.55           -0.27                0.21                0.76

            cmp to     z    p  -log2(p)
covariate                              
feature_0     0.00  0.56 0.58      0.80
feature_1     0.00  0.78 0.44      1.19
feature_2     0.00 -0.15 0.88      0.18
feature_3     0.00  0.00 1.00      0.00
feature_4     0.00  0.00 1.00      0.00
feature_5     0.00  0.00 1.00      0.00
feature_6     0.00  0.00 1.00      0.00
feature_9     0.00 -1.55 0.12      3.04
feature_10    0.00 -2.16 0.03      5.01
feature_11    0.00  0.79 0.43      1.22
feature_12    0.00  0.93 0.35      1.51
feature_20    0.00 -1.47 0.14      2.83
feature_21    0.00 -2.40 0.02      5.91
feature_22    0.00 -2.28 0.02      5.45
feature_23    0.00 -0.47 0.64      0.65
feature_24    0.00 -0.35 0.73      0.46
feature_29    0.00  0.00 1.00      0.00
feature_30    0.00 -0.00 1.00      0.00
feature_32    0.00 -2.78 0.01      7.54
---
Concordance = 0.70
Partial AIC = 571.45
log-likelihood ratio test = 48.85 on 19 df
-log2(p) of ll-ratio test = 12.34

Proportional hazard assumption looks okay.


[]

Now I know that the CoxPH assumptions are checked after the model is fitted.The CoxPH model makes several assumptions (especially the proportional hazards assumption), and you can only test these once you’ve estimated the model — because the tests and residuals depend on the fitted parameters.

- 3 main assumptions:
    - **1. Proportional hazards (PH) -->** The hazard ratios between groups are constant over time — i.e. the effect of covariates does not change with time. --> Schoenfeld residuals, time-varying tests, log-minus-log plots
    - **2. Linearity of continuous covariates -->** The log hazard is a linear function of continuous covariates. --> Martingale residual plots
    - **3. Independence of survival times and censoring -->** Censoring should be non-informative (not related to the event process). --> Study design assumption — not testable from the data alone.